# Работа с базами данных (2)

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* Макрушин С.В. Лекция "Работа с базами данных"
* https://sqliteonline.com/
* https://docs.python.org/3/library/sqlite3.html
* https://www.sqlitetutorial.net/sqlite-index/
* https://docs.python.org/3/library/sqlite3.html#sqlite3.IntegrityError
* https://www.sqlitetutorial.net/sqlite-alter-table/
* https://www.sqlitetutorial.net/sqlite-create-view/
* https://habr.com/ru/post/664000/
* https://learnsql.com/blog/what-is-common-table-expression/


## Задачи для совместного разбора

In [2]:
import pandas as pd
import sqlite3

# данные
students = pd.DataFrame(
    [
        ("Сотников Евгений Янович", 1),
        ("Степанова Виктория Константиновна", 1),
        ("Горелова Вероника Яновна", 2),
        ("Гришин Иван Романович", 3),
    ],
    columns=["name", "group_id"],
)
groups = list(zip([1, 2, 3], ["ПМ20-1", "ПМ20-2", "ПМ20-3"]))

con = sqlite3.connect("demo.sqlite")
con.execute("PRAGMA foreign_keys = 1")
cur = con.cursor()

# создаем таблицы
sql = """
DROP TABLE IF EXISTS StudentGroup;
DROP TABLE IF EXISTS Student;
CREATE TABLE StudentGroup (
    id int PRIMARY KEY,
    name varchar
);

CREATE TABLE Student(
    name VARCHAR PRIMARY KEY,
    group_id INT,
    FOREIGN KEY (group_id) REFERENCES StudentGroup(id)
);
"""
cur.executescript(sql)
con.commit()

# добавляем записи
sql = """
INSERT INTO StudentGroup(id, name) VALUES (?, ?)
"""
cur.executemany(sql, groups)
con.commit()

students.to_sql("Student", con, if_exists="append", index=False)

1\. Добавить столбец Age со значением по умолчанию. Добавить запись к таблицу

In [3]:
sql = '''
ALTER TABLE Student ADD COLUMN age INT DEFAULT 18; 
'''
# ALTER TABLE - изменить таблицу Student добавить столбец age типа INT со значением по умолчанию 18
cur.execute(sql)
con.commit()

In [4]:
pd.read_sql_query("SELECT * FROM Student", con)

,name,group_id,age
0,Сотников Евгений Янович,1,18
1,Степанова Виктория Константиновна,1,18
2,Горелова Вероника Яновна,2,18
3,Гришин Иван Романович,3,18


In [5]:
sql = '''
INSERT INTO Student(name, group_id, age) VALUES ("Бокач Елена Владимировна", 3, 19);
'''
try:
    cur.execute(sql)
except sqlite3.OperationalError as e:
    print(e)
else:
    print('Ошибки нет')
    con.commit()

Ошибки нет


2\. Занумеруйте студентов в рамках каждой группы.

In [7]:
sql = '''
SELECT name,
        group_id,
        ROW_NUMBER() OVER() as row_name, 
        ROW_NUMBER() OVER(PARTITION BY group_id ORDER BY name DESC) as row_group 
FROM Student;
'''
# нумерация строк
# нумерация строк внутри группы, сортировка по имени в обратном порядке, т.е. от Z до А,
# PARTITION BY - разделить на группы по group_id и внутри каждой группы нумеровать строки
pd.read_sql_query(sql, con)

,name,group_id,row_name,row_group
0,Степанова Виктория Константиновна,1,1,1
1,Сотников Евгений Янович,1,2,2
2,Горелова Вероника Яновна,2,3,1
3,Гришин Иван Романович,3,4,1
4,Бокач Елена Владимировна,3,5,2


3\. Выведите уникальные номера студентов

In [8]:
sql = '''
SELECT DISTINCT row_num_gr
FROM (
    SELECT name,
        group_id,
        ROW_NUMBER() OVER(PARTITION BY group_id ORDER BY name DESC) as row_num_gr
    FROM Student
);
'''
# DISTINCT - уникальные значения
pd.read_sql_query(sql, con)

,row_num_gr
0,1
1,2


In [9]:
sql = '''
CREATE VIEW StudentWithView AS
    SELECT name,
        group_id,
        ROW_NUMBER() OVER(PARTITION BY group_id ORDER BY name DESC) as row_num_gr
    FROM Student
'''
# CREATE VIEW - виртуальное представление
con.execute(sql)
con.commit()

In [10]:
pd.read_sql_query("SELECT DISTINCT row_num_gr FROM StudentWithView", con)

,row_num_gr
0,1
1,2


In [11]:
sql = '''
WITH StudentWithNumberCTE AS (
    SELECT name,
        group_id,
        ROW_NUMBER() OVER(PARTITION BY group_id ORDER BY name DESC) as row_num_gr
    FROM Student
)
SELECT DISTINCT row_num_gr
FROM StudentWithNumberCTE
'''

pd.read_sql_query(sql, con)

,row_num_gr
0,1
1,2


## Лабораторная работа 4

__При решении данных задач не подразумевается использования циклов или генераторов Python в ходе работы с пакетами `numpy` и `pandas`, если в задании не сказано обратного. Решения задач, в которых для обработки массивов `numpy` или структур `pandas` используются явные циклы (без согласования с преподавателем), могут быть признаны некорректными и не засчитаны.__

__Для начала работы подключитесь к БД `recipes.db` и создайте объект-курсор.__

In [16]:
con = sqlite3.connect("recipes.db")
cur = con.cursor()

In [17]:
pd.read_sql_query("SELECT * FROM Review", con)

,id,user_id,recipe_id,date,rating,review
0,370476,21752,57993,2003-05-01,5,Last week whole sides of frozen salmon fillet ...
1,624300,431813,142201,2007-09-16,5,So simple and so tasty! I used a yellow capsi...
2,187037,400708,252013,2008-01-10,4,"Very nice breakfast HH, easy to make and yummy..."
3,706134,2001852463,404716,2017-12-11,5,These are a favorite for the holidays and so e...
4,312179,95810,129396,2008-03-14,5,Excellent soup! The tomato flavor is just gre...
...,...,...,...,...,...,...
126691,1013457,1270706,335534,2009-05-17,4,This recipe was great! I made it last night. I...
126692,158736,2282344,8701,2012-06-03,0,This recipe is outstanding. I followed the rec...
126693,1059834,689540,222001,2008-04-08,5,"Well, we were not a crowd but it was a fabulou..."
126694,453285,2000242659,354979,2015-06-02,5,I have been a steak eater and dedicated BBQ gr...


<p class="task" id="1"></p>

1\. Создайте уникальный индекс для таблицы `Review` для обеспечения уникальности сочетания значений в полях `user_id` и `recipe_id`. 

In [19]:
sql = '''
CREATE UNIQUE INDEX IF NOT EXISTS index_review ON Review (recipe_id, user_id);
'''

cur.execute(sql)
con.commit()


In [20]:
pd.read_sql_query("SELECT * FROM Review", con)

,id,user_id,recipe_id,date,rating,review
0,370476,21752,57993,2003-05-01,5,Last week whole sides of frozen salmon fillet ...
1,624300,431813,142201,2007-09-16,5,So simple and so tasty! I used a yellow capsi...
2,187037,400708,252013,2008-01-10,4,"Very nice breakfast HH, easy to make and yummy..."
3,706134,2001852463,404716,2017-12-11,5,These are a favorite for the holidays and so e...
4,312179,95810,129396,2008-03-14,5,Excellent soup! The tomato flavor is just gre...
...,...,...,...,...,...,...
126691,1013457,1270706,335534,2009-05-17,4,This recipe was great! I made it last night. I...
126692,158736,2282344,8701,2012-06-03,0,This recipe is outstanding. I followed the rec...
126693,1059834,689540,222001,2008-04-08,5,"Well, we were not a crowd but it was a fabulou..."
126694,453285,2000242659,354979,2015-06-02,5,I have been a steak eater and dedicated BBQ gr...


<p class="task" id="2"></p>

2\. Напишите функцию `add_review(review_id, user_id, recipe_id, date, rating, review)`, которая добавляет запись в таблицу `Review`. В случае успешного добавления функция должна вернуть значение 0. В случае нарушения ограничения целостности функция должна вернуть значение 1. В случае любых других ошибок функция должна вернуть значение 2. Продемонстрируйте работу функции, попытавшись добавить одну и ту же запись дважды в двух ячейках подряд.

Для решения задачи воспользуйтесь механизмом try - except и обработайте соответствующее исключение.

In [27]:
#IntegrityError : Исключение sqlite3. IntegrityError возникает, когда затрагивается реляционная целостность базы данных, 
#например когда проверка внешнего ключа не удалась.

In [28]:
#Исключение sqlite3.ProgrammingError возникает из за ошибки программирования, например:

#таблица не найдена или уже существует,
#синтаксическая ошибка в операторе SQL,
#неверное количество указанных параметров и т. д.

In [29]:

#Исключение sqlite3.OperationalError возникает при ошибках, связанных с работой базы данных и не обязательно находятся под контролем программиста, например:

#происходит неожиданное отключение,
#имя источника данных не найдено,
#транзакция не может быть обработана и т. д.

In [30]:
#sqlite3.NotSupportedError:
#Исключение sqlite3.NotSupportedError 
#Возникает в случае использования метода или API, который не поддерживается базой данных, например:
#вызов метода connect.rollback() для соединения, которое не поддерживает транзакции или когда транзакции отключены.


In [21]:
def add_review(review_id, user_id, recipe_id, date, rating, review):
    try:
        sql = '''
        INSERT INTO Review(id, user_id, recipe_id, date, rating, review) 
        VALUES (?, ?, ?, ?, ?, ?)
        '''
        cur.execute(sql, (review_id, user_id, recipe_id, date, rating, review))
        con.commit()
        return 0
    except sqlite3.IntegrityError:
        return 1
    except sqlite3.Error:
        return 2

In [22]:
add_review(1, 1, 1, '2021-01-01', 5, 'Great recipe!')

0

In [23]:
pd.read_sql_query("SELECT * FROM Review WHERE id = 1", con)

,id,user_id,recipe_id,date,rating,review
0,1,1,1,2021-01-01,5,Great recipe!


In [24]:
add_review(1, 1, 1, '2021-01-01', 5, 'Great recipe!')

1

In [25]:
def add_review(review_id, user_id, recipe_id, date, rating, review):
    try:
        sql = '''
        INSERT INTO Reviews(id, user_id, recipe_id, date, rating, review) 
        VALUES (?, ?, ?, ?, ?, ?)
        '''
        cur.execute(sql, (review_id, user_id, recipe_id, date, rating, review))
        con.commit()
        return 0
    except sqlite3.IntegrityError:
        return 1
    except sqlite3.Error:
        return 2

In [26]:
add_review(2, 2, 2, '2020-01-01', 4, 'Good recipe!=')

2

<p class="task" id="3"></p>

3\. _Измените_ таблицу Review, добавив в нее поле `toxic` булева типа. 

In [31]:
sql = '''
ALTER TABLE Review ADD COLUMN toxic BOOLEAN DEFAULT None;
'''
cur.execute(sql)
con.commit()

In [32]:
pd.read_sql_query("SELECT * FROM Review", con)

,id,user_id,recipe_id,date,rating,review,toxic
0,370476,21752,57993,2003-05-01,5,Last week whole sides of frozen salmon fillet ...,None
1,624300,431813,142201,2007-09-16,5,So simple and so tasty! I used a yellow capsi...,None
2,187037,400708,252013,2008-01-10,4,"Very nice breakfast HH, easy to make and yummy...",None
3,706134,2001852463,404716,2017-12-11,5,These are a favorite for the holidays and so e...,None
4,312179,95810,129396,2008-03-14,5,Excellent soup! The tomato flavor is just gre...,None
...,...,...,...,...,...,...,...
126692,158736,2282344,8701,2012-06-03,0,This recipe is outstanding. I followed the rec...,None
126693,1059834,689540,222001,2008-04-08,5,"Well, we were not a crowd but it was a fabulou...",None
126694,453285,2000242659,354979,2015-06-02,5,I have been a steak eater and dedicated BBQ gr...,None
126695,691207,463435,415599,2010-09-30,5,Wonderful and simple to prepare seasoning blen...,None


<p class="task" id="4"></p>

4\. Вам дан классификатор `clf`, который классифицирует тексты отзывов как токсичные (`True`) и не токсичные (`False`).
Напишите функцию `classify_reviews`, которая итеративно получает пакет (батч) `batch_size` строк из таблицы Reviews, у которых не проставлено значение в столбце `toxic`, делает для них прогноз при помощи модели `clf` и обновляет соответствующие строки в БД. Данная процедура выполняется до тех пор, пока в БД есть строки, для которых требуется получить прогноз.

Продемонстрируйте результат, выведя на экран количество токсичных и не токсичных отзывов в таблице.

In [33]:
from sklearn.dummy import DummyClassifier

clf = DummyClassifier(strategy="uniform").fit(None, [True, False])

In [ ]:
def classify_reviews(batch_size=10000):
    sql = '''
    
    '''

<p class="task" id="5"></p>

5\. Создайте представление `RecipeWithYear`, в котором добавлен дополнительный столбец `year`, содержащий год даты из столбца `submitted`. Сделайте выборку из этого представления и выведите на экран количество рецептов с разбивкой по годам.

In [34]:
#strftime()	Возвращает строку, отформатированную в соответствии с указанным форматом.
#%Y	- год с 4 цифрами
#%y - год с 2 цифрами

In [35]:
sql = '''
CREATE VIEW RecipeWithYear AS 
SELECT *, strftime('%Y', submitted) as year
FROM Recipe
'''
cur.execute(sql)

In [37]:
sql = '''
SELECT year, COUNT(*) as cnt
FROM RecipeWithYear
GROUP BY year
'''
cur.execute(sql)

In [38]:
cur.fetchall()

[('1999', 275),
 ('2000', 104),
 ('2001', 589),
 ('2002', 2644),
 ('2003', 2334),
 ('2004', 2153),
 ('2005', 3130),
 ('2006', 3473),
 ('2007', 4429),
 ('2008', 4029),
 ('2009', 2963),
 ('2010', 1538),
 ('2011', 922),
 ('2012', 659),
 ('2013', 490),
 ('2014', 139),
 ('2015', 42),
 ('2016', 24),
 ('2017', 39),
 ('2018', 24)]

<p class="task" id="6"></p>

6\. Напишите запрос на языке SQL, который возвращает все строки из таблицы `Recipe` с дополнительным столбцом, содержащем номер рецепта. Рецепты нумеруются целыми числами, начиная с 1, в __рамках каждого года__ в порядке их добавления в БД (столбец `submitted`). Получите результат в виде `pd.DataFrame`. Посчитайте и выведите на экран количество строк полученного `pd.DataFrame`, для которых сгенерированный номер кратен 50.

In [41]:
sql = '''
SELECT *, ROW_NUMBER() OVER (PARTITION BY year ORDER BY submitted) AS num_row FROM RecipeWithYear
'''

result = pd.read_sql_query(sql, con)
result.head()

,id,name,minutes,submitted,description,n_ingredients,year,num_row
0,203,chinese plum sauce,115,1999-08-06,chinese plum sauce serve this with egg rolls ...,12.0,1999,1
1,653,b c cherry and raspberry preserves,215,1999-08-09,None,4.0,1999,2
2,360,baked zucchini frittatas,67,1999-08-09,None,NaN,1999,3
3,658,dried fruit roll ups,1495,1999-08-09,fruit roll-ups,4.0,1999,4
4,1144,steak tomato basil pasta,0,1999-08-09,None,11.0,1999,5


In [42]:
result = result[result.num_row%50==0]
result.shape[0]

589

<p class="task" id="7"></p>

7\. Используя обобщенное табличное выражение и решение задачи 6, напишите запрос на языке SQL, который вернет количество строк, для которых сгенерированный номер кратен 50. Выполните запрос и выведите количество таких строк на экран.

In [43]:
sql = '''
SELECT COUNT(*) as cnt
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY year ORDER BY submitted) as num FROM RecipeWithYear
) WHERE num % 50 = 0
'''
cur.execute(sql)
cur.fetchone()[0]

589